# Forex RL experiment

Thin interface for the reusable `forex_rl` package. Training and complete walk-forward evaluation are explicit opt-in operations.

In [ ]:
# pip install -e .[research]
from forex_rl import (ExperimentConfig, download_eurusd, compute_indicators, build_walkforward_folds, make_scaled_env, make_model, train_one_fold, backtest_model, run_baselines, run_walkforward, generate_trade_report, generate_walkforward_report, zip_directory)

config = ExperimentConfig()
feature_cols = ['ema20', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower', 'atr_n', 'ret']
price_cols = ['open', 'high', 'low', 'close', 'atr']

In [ ]:
# Opt-in external data acquisition; no model training occurs here.
raw = download_eurusd(config.symbol, config.start_date, config.end_date)
data = compute_indicators(raw)
folds = build_walkforward_folds(data, config.min_train_months, config.val_months, config.test_months, config.step_months, config.use_rolling, config.rolling_train_months)
print(f'Rows: {len(data):,}; folds: {len(folds)}')

In [ ]:
# Leakage-safe SB3 environment setup. The scaler is fit only on train features.
fold = folds[0]
train = data.loc[fold['train_start']:fold['train_end']].copy()
validation = data.loc[fold['val_start']:fold['val_end']].copy()
train_env, validation_env, scaler = make_scaled_env(train, validation, feature_cols, price_cols, config.environment_kwargs, config.random_seed, n_envs=1)
# model = make_model(config.algo, train_env, config)  # construction only; model.learn is deliberately opt-in

In [ ]:
# Explicit training hook. Do not run this as routine refactor validation.
RUN_FULL_WALKFORWARD = False
if RUN_FULL_WALKFORWARD:
    results = run_walkforward(config.algo, data, folds, config, train_one_fold, feature_cols, price_cols)
    report = generate_walkforward_report(results, config.report_dir, config.algo)
    # Optional local archive: zip_directory(config.report_dir, f'{config.log_dir}/reports_pack.zip')
    print(report['html'])

The package also exposes deterministic `backtest_model`, baseline comparisons via `run_baselines`, R-based model selection helpers, and `generate_trade_report`. Existing trading semantics and known limitations are documented in `docs/STEP1_CURRENT_IMPLEMENTATION_MAP.md`.